In [1]:
import pandas as pd

# load todas bases iptu com bairro e cep


In [2]:

import chardet
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
years = range(2021, 2025)

dfs = []
selected_columns = ["NUMERO DO CONTRIBUINTE",
                    "ANO DO EXERCICIO",
                    "NOME DE LOGRADOURO DO IMOVEL",
                    "NUMERO DO IMOVEL",
                    "AREA DO TERRENO",
                    "AREA CONSTRUIDA",
                    "AREA OCUPADA",
                    "VALOR DO M2 DO TERRENO",
                    "VALOR DO M2 DE CONSTRUCAO",
                    "TIPO DE USO DO IMOVEL",
                    "TIPO DE PADRAO DA CONSTRUCAO",
                    "BAIRRO DO IMOVEL",
                    "CEP DO IMOVEL"
                    ]

for year in years:
    file_path = f"data/IPTU_{year}.csv"
    try:
        if year < 2022:
            df = pd.read_csv(file_path, sep=';', encoding='utf-8-sig', low_memory=False)
        else:
            df = pd.read_csv(file_path, sep=';', encoding='latin-1', low_memory=False)
        if "ï»¿NUMERO DO CONTRIBUINTE" in df.columns:
            df = df.rename(columns={"ï»¿NUMERO DO CONTRIBUINTE": "NUMERO DO CONTRIBUINTE"})
        df = df[selected_columns]
        if df['VALOR DO M2 DO TERRENO'].dtype == "object":
            df['VALOR DO M2 DO TERRENO'] = df['VALOR DO M2 DO TERRENO'].str.replace(',', '.')
        if df['VALOR DO M2 DE CONSTRUCAO'].dtype == "object":
            df['VALOR DO M2 DE CONSTRUCAO'] = df['VALOR DO M2 DE CONSTRUCAO'].str.replace(',', '.')
        dfs.append(df)
        print(f"Successfully read {file_path}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

File not found: drive/MyDrive/IPTU_2021.csv
File not found: drive/MyDrive/IPTU_2022.csv
File not found: drive/MyDrive/IPTU_2023.csv
File not found: drive/MyDrive/IPTU_2024.csv


In [ ]:
if dfs:
    df_combined = pd.concat(dfs, ignore_index=True)
    print("\nCombined DataFrame:")
    display(df_combined.head())
    display(df_combined.info())
else:
    print("No dataframes were loaded.")

del dfs


Combined DataFrame:


,NUMERO DO CONTRIBUINTE,ANO DO EXERCICIO,NOME DE LOGRADOURO DO IMOVEL,NUMERO DO IMOVEL,AREA DO TERRENO,AREA CONSTRUIDA,AREA OCUPADA,VALOR DO M2 DO TERRENO,VALOR DO M2 DE CONSTRUCAO,TIPO DE USO DO IMOVEL,TIPO DE PADRAO DA CONSTRUCAO,BAIRRO DO IMOVEL,CEP DO IMOVEL
0,0010030001-4,2021,R S CAETANO,13.00,136,135.00,108.00,2319.00,1726.00,Loja,Comercial horizontal - padrão B,SANTA EFIGENIA,01104-001
1,0010030002-2,2021,R S CAETANO,19.00,90,67.00,67.00,2319.00,1726.00,Loja,Comercial horizontal - padrão B,SANTA EFIGENIA,01104-001
2,0010030003-0,2021,R S CAETANO,27.00,105,140.00,84.00,2319.00,1726.00,Loja,Comercial horizontal - padrão B,SANTA EFIGENIA,01104-001
3,0010030004-9,2021,R S CAETANO,33.00,108,103.00,86.00,2319.00,1726.00,Loja,Comercial horizontal - padrão B,NaN,01104-001
4,0010030005-7,2021,R S CAETANO,39.00,120,170.00,110.00,2319.00,1726.00,Loja,Comercial horizontal - padrão B,NaN,01104-001


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14371561 entries, 0 to 14371560
Data columns (total 13 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   NUMERO DO CONTRIBUINTE        object 
 1   ANO DO EXERCICIO              int64  
 2   NOME DE LOGRADOURO DO IMOVEL  object 
 3   NUMERO DO IMOVEL              float64
 4   AREA DO TERRENO               int64  
 5   AREA CONSTRUIDA               float64
 6   AREA OCUPADA                  float64
 7   VALOR DO M2 DO TERRENO        object 
 8   VALOR DO M2 DE CONSTRUCAO     object 
 9   TIPO DE USO DO IMOVEL         object 
 10  TIPO DE PADRAO DA CONSTRUCAO  object 
 11  BAIRRO DO IMOVEL              object 
 12  CEP DO IMOVEL                 object 
dtypes: float64(3), int64(2), object(8)
memory usage: 1.4+ GB


None

In [ ]:
df_combined.to_csv('drive/MyDrive/IPTU_2019_2024_mod.csv', index=False)

# gerar base de bairro por setor

In [ ]:
#base = pd.read_csv('drive/MyDrive/IPTU_2024.csv', sep=';', encoding='latin-1')
base = pd.read_csv('drive/MyDrive/IPTU_2019_2024_mod.csv', sep=',')

In [ ]:
base['NUMERO DO CONTRIBUINTE'] = base['NUMERO DO CONTRIBUINTE'].astype(str).str.replace('-', '')
base['NUMERO DO CONTRIBUINTE'] = base['NUMERO DO CONTRIBUINTE'].str.zfill(11)
base['setor'] = base['NUMERO DO CONTRIBUINTE'].astype(str).str[:3]

In [ ]:
base_bairro = (
    base
    .groupby('setor', as_index = False)
    .agg(bairro = ('BAIRRO DO IMOVEL', lambda x: x.mode().iloc[0] if not x.mode().empty else None))
)

In [ ]:
base_bairro

,setor,bairro
0,001,CENTRO
1,002,BRAS
2,003,BRAS
3,004,CAMBUCI
4,005,LIBERDADE
...,...,...
294,306,CASA VERDE
295,307,FREGUESIA DO O
296,308,JD PAULISTANO
297,309,SAUDE


In [ ]:
base_bairro.loc[base_bairro.bairro.isnull()]

,setor,bairro
202,204,None
216,224,None
284,295,None


In [ ]:
base_bairro.to_csv('drive/MyDrive/base_bairro_setor.csv', index=False)

In [ ]:
pd.set_option('display.max_columns', None)

# juntando com bases

## itbi

In [ ]:
itbi = pd.read_csv('drive/MyDrive/ITBI_2019_2024.csv', sep=',')

/tmp/ipython-input-1141025404.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  itbi = pd.read_csv('drive/MyDrive/ITBI_2019_2024.csv', sep=',')


In [ ]:
itbi['N° do Cadastro (SQL)'] = itbi['N° do Cadastro (SQL)'].astype(str).str.replace('-', '')
itbi['N° do Cadastro (SQL)'] = itbi['N° do Cadastro (SQL)'].astype(str).str.zfill(11)
itbi['setor'] = itbi['N° do Cadastro (SQL)'].astype(str).str[:3]
itbi['quadra'] = itbi['N° do Cadastro (SQL)'].astype(str).str[3:6]
itbi['lote'] = itbi['N° do Cadastro (SQL)'].astype(str).str[6:10]

In [ ]:
itbi = (
    itbi
    .merge(
        base_bairro,
        on = 'setor',
        how = 'left'
        )
)

In [ ]:
itbi = itbi.drop(index=744253) # Tinha uma linha toda cagada

In [ ]:
itbi.loc[itbi.bairro.isnull()]

,N° do Cadastro (SQL),Natureza de Transação,Valor de Transação (declarado pelo contribuinte),Data de Transação,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Tipo de Financiamento,Valor Financiado,Situação do SQL,Ano da Transacao,Mes da Transacao,Ano/Mes da Transacao,setor,quadra,lote,bairro


In [ ]:
itbi.to_csv('drive/MyDrive/ITBI_2019_2024_v2.csv', index=False)

In [ ]:
itbi

,N° do Cadastro (SQL),Natureza de Transação,Valor de Transação (declarado pelo contribuinte),Data de Transação,Valor Venal de Referência,Proporção Transmitida (%),Valor Venal de Referência (proporcional),Base de Cálculo adotada,Tipo de Financiamento,Valor Financiado,Situação do SQL,Ano da Transacao,Mes da Transacao,Ano/Mes da Transacao,setor,quadra,lote,bairro
0,12310100659,1.Compra e venda,900000.00,2019-01-02,773836.0,100.00,773836.0000,900000.00,1.Sistema Financeiro de Habitação,400000.00,Ativo Predial,2019,1,2019/01,123,101,0065,JARDIM LEONOR
1,05528502403,1.Compra e venda,420000.00,2019-01-02,303832.0,100.00,303832.0000,420000.00,1.Sistema Financeiro de Habitação,340721.07,Ativo Predial,2019,1,2019/01,055,285,0240,VL FORMOSA
2,29903601032,1.Compra e venda,650000.00,2019-01-03,459364.0,100.00,459364.0000,650000.00,NaN,0.00,Ativo Predial,2019,1,2019/01,299,036,0103,VILA OLIMPIA
3,04201104559,1.Compra e venda,61765.36,2019-01-03,300840.0,100.00,300840.0000,300840.00,NaN,0.00,Ativo Predial,2019,1,2019/01,042,011,0455,SAUDE
4,02206602970,1.Compra e venda,965000.00,2019-01-03,575501.0,100.00,575501.0000,965000.00,NaN,0.00,Ativo Predial,2019,1,2019/01,022,066,0297,PERDIZES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941619,03302900651,2.Cessão de direitos relativos a compromisso d...,89372.68,2024-12-30,1564307.0,0.62,9698.7034,89372.68,NaN,0.00,Ativo Territorial,2024,12,2024/12,033,029,0065,ACLIMACAO
941620,03302900651,2.Cessão de direitos relativos a compromisso d...,89372.68,2024-12-30,1564307.0,0.62,9698.7034,89372.68,NaN,0.00,Ativo Territorial,2024,12,2024/12,033,029,0065,ACLIMACAO
941621,03302900651,2.Cessão de direitos relativos a compromisso d...,67337.54,2024-12-30,1564307.0,0.46,7195.8122,67337.54,NaN,0.00,Ativo Territorial,2024,12,2024/12,033,029,0065,ACLIMACAO
941622,03302900651,2.Cessão de direitos relativos a compromisso d...,89372.68,2024-12-30,1564307.0,0.62,9698.7034,89372.68,NaN,0.00,Ativo Territorial,2024,12,2024/12,033,029,0065,ACLIMACAO


## iptu

In [ ]:
iptu = pd.read_csv('drive/MyDrive/IPTU_2019_2024.csv', sep=',')

In [ ]:
iptu['NUMERO DO CONTRIBUINTE'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str.replace('-', '')
iptu['NUMERO DO CONTRIBUINTE'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str.zfill(11)
iptu['setor'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str[:3]
iptu['quadra'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str[3:6]
iptu['lote'] = iptu['NUMERO DO CONTRIBUINTE'].astype(str).str[6:10]

In [ ]:
iptu = (
    iptu
    .merge(
        base_bairro,
        on = 'setor',
        how = 'left'
        )
)

In [ ]:
iptu